# Avance 2 — API Yelp (Simulada/Real)

**Estudiante:** Lourdes Diamela Alarcon

_Actualizado: 2025-09-17_

## Descripción
Se obtiene información de negocios locales desde **Yelp** usando API (si existe `YELP_API_KEY`) con respaldo en CSV. Luego se integran agregados por ciudad al dataset de clientes. Se añade una señal de **tendencia externa** mediante scraping con respaldo.


In [ ]:

import os, requests
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

CLIENTES_CITY = '../data/processed_clientes_city.csv'
YELP_CSV = '../data/yelp_restaurants.csv'

df_clients = pd.read_csv(CLIENTES_CITY)
print('Clientes (ciudad):', df_clients.shape)
df_clients.head(3)


### Conexión a Yelp (opcional) con `YELP_API_KEY` y respaldo CSV

In [ ]:

yelp_api_key = os.environ.get('YELP_API_KEY','')

def fetch_yelp(city='Miami', limit=30):
    if not yelp_api_key:
        raise RuntimeError('Falta YELP_API_KEY; se usará CSV.')
    url = 'https://api.yelp.com/v3/businesses/search'
    headers = {'Authorization': f'Bearer {yelp_api_key}'}
    params = {'location': city, 'categories': 'restaurants', 'limit': limit}
    r = requests.get(url, headers=headers, params=params, timeout=15)
    r.raise_for_status()
    js = r.json()
    rows = []
    for b in js.get('businesses', []):
        rows.append({
            'name': b.get('name'),
            'rating': b.get('rating'),
            'review_count': b.get('review_count'),
            'price': b.get('price', 'No especificado'),
            'city': (b.get('location') or {}).get('city'),
            'coordinates_latitude': (b.get('coordinates') or {}).get('latitude'),
            'coordinates_longitude': (b.get('coordinates') or {}).get('longitude'),
            'location_address1': (b.get('location') or {}).get('address1'),
            'categories': ', '.join([c.get('title','') for c in b.get('categories',[])])
        })
    return pd.DataFrame(rows)

try:
    ciudad_obj = df_clients['ciudad_residencia'].mode().iat[0] if 'ciudad_residencia' in df_clients.columns else 'Miami'
except Exception:
    ciudad_obj = 'Miami'

try:
    df_yelp_live = fetch_yelp(ciudad_obj, 30)
    yelp_df = df_yelp_live
    print('Yelp API (on):', yelp_df.shape)
except Exception as e:
    print('Yelp API no disponible ->', e, '\nSe utiliza CSV local.')
    yelp_df = pd.read_csv(YELP_CSV)

yelp_df.head(3)


### Selección de columnas y normalización

In [ ]:

keep = [c for c in ['name','alias','title','price','rating','review_count','city','coordinates_latitude','coordinates_longitude','location_address1','categories'] if c in yelp_df.columns]
yelp_sel = yelp_df[keep].copy()
yelp_sel['city_norm'] = yelp_sel.get('city','').astype(str).str.strip().str.lower()

df_clients['city_norm'] = df_clients['ciudad_residencia'].astype(str).str.strip().str.lower() if 'ciudad_residencia' in df_clients.columns else ''
ciudad_obj = df_clients['city_norm'].mode().iat[0] if 'city_norm' in df_clients.columns and len(df_clients) else 'miami'

yelp_city = yelp_sel[yelp_sel['city_norm']==ciudad_obj].copy()
print('Ciudad objetivo:', ciudad_obj, '| Yelp filas:', yelp_city.shape[0])
yelp_city.head(3)


### Agregados de Yelp por ciudad e integración con clientes

In [ ]:

agg_rating = yelp_city['rating'].mean() if 'rating' in yelp_city.columns else np.nan
agg_reviews_med = yelp_city['review_count'].median() if 'review_count' in yelp_city.columns else np.nan

df_enriched = df_clients.copy()
df_enriched['yelp_city'] = ciudad_obj
df_enriched['yelp_rating_ciudad_prom'] = agg_rating
df_enriched['yelp_mediana_reviews_ciudad'] = agg_reviews_med

df_enriched.to_csv('../data/processed_clientes_enriched.csv', index=False)
print('Exportado: processed_clientes_enriched.csv')


### Tendencias externas vía scraping (con respaldo)

In [ ]:

import datetime as dt

def scrape_us_holidays(default_city='Miami'):
    import requests
    from bs4 import BeautifulSoup
    url = 'https://en.wikipedia.org/wiki/Federal_holidays_in_the_United_States'
    html = requests.get(url, timeout=15).text
    soup = BeautifulSoup(html, 'html.parser')
    tables = soup.find_all('table', {'class':'wikitable'})
    rows = []
    for tr in tables[0].find_all('tr')[1:]:
        tds = tr.find_all(['td','th'])
        txt = [td.get_text(strip=True) for td in tds]
        if len(txt) >= 2:
            rows.append({'holiday': txt[0], 'when': txt[1]})
    out = pd.DataFrame(rows)
    out['city'] = default_city
    out['trend_index'] = 60
    return out

try:
    default_city = df_enriched['ciudad_residencia'].mode().iat[0] if 'ciudad_residencia' in df_enriched.columns else 'Miami'
    df_hol = scrape_us_holidays(default_city)
    print('Scraping OK:', df_hol.shape)
except Exception as e:
    print('Scraping no disponible ->', e, '\nSe usa fallback local.')
    fallback = '../data/trends_miami_sample.csv'
    if not Path(fallback).exists():
        dates = pd.date_range('2025-04-01', periods=16, freq='W')
        pd.DataFrame({'city':'Miami','date':dates.date,'trend_index':50}).to_csv(fallback, index=False)
    df_hol = pd.read_csv(fallback).rename(columns={'date':'when'})

trend_city_mean = df_hol.groupby('city')['trend_index'].mean().to_dict() if 'trend_index' in df_hol.columns else {df_hol['city'].iat[0]:50}
df_enriched_trends = df_enriched.copy()
df_enriched_trends['trend_index_city_mean'] = df_enriched_trends['city_norm'].map(lambda c: trend_city_mean.get(c, np.nan))

df_enriched_trends.to_csv('../data/processed_clientes_enriched_trends.csv', index=False)
print('Exportado: processed_clientes_enriched_trends.csv')
